# KARE on SynVG-ZS — reproduction notebook

Reproduces every number, table and figure in the paper *KARE: Knowledge-Anchored Predicate Prototypes and Coverage-Aware Evidential Fusion for Zero-Shot Scene Graph Generation*. CPU only; needs NumPy, SciPy, autograd and pandas.

| Paper item | Produced by |
|---|---|
| Table 1 (worked fusion example) | Section 5 (`extra_analyses.py`) |
| Table 2 (benchmark statistics, seed 0), Table 13 (per seed) | Sections 2 and 5 |
| Table 3 (hyper-parameters; selected values) | Section 5 |
| Tables 4–9 (main results, bootstrap, ablations, per-predicate, coverage bins) | Section 3 (`aggregate.py`) |
| Tables 10–11 (seen/unseen breakdown, τ curve), Spearman test in §5.3 | Section 5 |
| Figures 7–8 (robustness sweeps) and the genericity-zero ablation (§5.2) | Section 4 |
| Label-noise analysis (§5.7) | Section 5 |
| Figures 2–11 as publication PNGs | Section 6 (`figures.py`) |

Runtime on one CPU core: about 8 minutes for Section 2 and about 30 minutes for Section 4. All randomness comes from one NumPy generator per run, seeded by the run's seed.

## 0. Setup

In [ ]:
!pip install -q autograd pandas scipy
import os; os.makedirs('out', exist_ok=True)

## 1. Code modules
`kare.py`: world, data, KG, models, KAPT, fusion rules and metrics. `run.py`: one seed end to end (all baselines, ablations and pseudo-zero-shot tuning). `aggregate.py`: Tables 4–9 and the paired bootstrap. `sweep.py` / `agg_sweep.py`: robustness sweeps. `extra_analyses.py`: Tables 1, 3, 10, 11, 13 and the Spearman test.

In [ ]:
%%writefile kare.py
"""KARE on SynVG-ZS — implementation of Sections 3-4 of the paper.

Everything (world, dataset, KG, models, metrics) is drawn from one seed.
"""
import numpy as onp
import autograd.numpy as anp
from autograd import grad
from scipy.special import logsumexp, softmax as sp_softmax

# --------------------------------------------------------------------------- constants
VG50 = ["on", "has", "wearing", "of", "in", "near", "behind", "with", "holding", "above",
        "sitting on", "wears", "under", "riding", "in front of", "standing on", "at", "carrying",
        "attached to", "walking on", "over", "for", "looking at", "watching", "hanging from",
        "laying on", "eating", "and", "belonging to", "parked on", "using", "covering", "between",
        "along", "covered in", "part of", "lying on", "on back of", "to", "walking in",
        "mounted on", "across", "against", "from", "growing on", "painted on", "playing",
        "made of", "says", "flying in"]
K = 50
UNSEEN = ["riding", "eating", "carrying", "covering", "looking at", "using", "hanging from",
          "playing", "walking on", "parked on"]
PSEUDO = ["sitting on", "standing on", "watching", "laying on", "mounted on", "walking in"]
GENERIC = ["on", "has", "of", "in", "near", "with"]
PARENT = {"riding": "on", "eating": "holding", "carrying": "holding", "covering": "on",
          "looking at": "near", "using": "holding", "hanging from": "on", "playing": "with",
          "walking on": "on", "parked on": "on", "sitting on": "on", "standing on": "on",
          "watching": "near", "laying on": "on", "mounted on": "on", "walking in": "in",
          "wearing": "has", "wears": "has", "holding": "has", "lying on": "on",
          "painted on": "on", "growing on": "on", "attached to": "on", "covered in": "with",
          "part of": "of", "belonging to": "of", "made of": "of", "on back of": "on",
          "flying in": "in"}
PID = {p: i for i, p in enumerate(VG50)}
U_IDX = onp.array([PID[p] for p in UNSEEN])
S_IDX = onp.array([i for i in range(K) if VG50[i] not in UNSEEN])
PS_IDX = onp.array([PID[p] for p in PSEUDO])

N_TYPES, PER_TYPE = 10, 15
N_CLS = N_TYPES * PER_TYPE
D_LAT, D_REG, D_GEO, D_UNI, D_LAB, D_JOINT = 24, 32, 8, 16, 16, 32

DEFAULT = dict(n_train=5000, n_val=800, n_test=1500, zipf=1.15, generic_rate=0.35,
               cov_lo=0.04, cov_hi=0.35, type_p=0.7, wrong_frac=0.10, nonvis=2500,
               text_noise=0.55, det_recall=0.92, confusion=0.15,
               kg_cov_mult=1.0, generic_mult=1.0)


# --------------------------------------------------------------------------- world
class World:
    def __init__(self, rng, cfg):
        self.cfg = cfg
        self.cls_type = onp.repeat(onp.arange(N_TYPES), PER_TYPE)
        tc = rng.normal(0, 1, (N_TYPES, D_LAT))
        self.lat = tc[self.cls_type] + rng.normal(0, 0.6, (N_CLS, D_LAT))
        self.W_reg = rng.normal(0, 1 / onp.sqrt(D_LAT), (D_REG, D_LAT))
        self.W_lab = rng.normal(0, 1 / onp.sqrt(D_LAT), (D_LAB, D_LAT))
        self.lab_emb = self.lat @ self.W_lab.T + rng.normal(0, 0.3, (N_CLS, D_LAB))
        # predicate signatures (child partly inherits parent's look)
        sig = rng.normal(0, 1, (K, D_UNI)); geo = rng.normal(0, 1, (K, D_GEO))
        for c, p in PARENT.items():
            sig[PID[c]] = 0.85 * sig[PID[c]] + 0.35 * sig[PID[p]]
            geo[PID[c]] = 0.85 * geo[PID[c]] + 0.35 * geo[PID[p]]
        self.sig, self.geo = sig, geo
        self.vis = onp.where(onp.isin(onp.arange(K), [PID[g] for g in GENERIC]), 0.25,
                             rng.uniform(0.45, 1.3, K))
        # text embeddings: noisy linear view of each predicate's visual signature
        P = rng.normal(0, 1, (D_JOINT, D_UNI + D_GEO))
        clean = onp.concatenate([sig, geo], 1) @ P.T
        noise = rng.normal(0, 1, clean.shape)
        noise *= (cfg["text_noise"] * onp.linalg.norm(clean, axis=1) / onp.linalg.norm(noise, axis=1))[:, None]
        t = clean + noise
        self.text = t / onp.linalg.norm(t, axis=1, keepdims=True)
        # predicate type compatibility and valid concept pairs
        self.allowed, self.valid = [], []
        for r in range(K):
            g = VG50[r] in GENERIC
            ns = rng.integers(4, 7) if g else rng.integers(1, 3)
            no = rng.integers(4, 7) if g else rng.integers(1, 4)
            st = rng.choice(N_TYPES, ns, replace=False); ot = rng.choice(N_TYPES, no, replace=False)
            pairs_t = [(a, b) for a in st for b in ot]
            self.allowed.append(pairs_t)
            vp = []
            for a, b in pairs_t:
                ca = onp.where(self.cls_type == a)[0]; cb = onp.where(self.cls_type == b)[0]
                m = rng.random((PER_TYPE, PER_TYPE)) < 0.35
                ii, jj = onp.nonzero(m)
                vp += [(ca[i], cb[j]) for i, j in zip(ii, jj) if ca[i] != cb[j]]
            self.valid.append(onp.array(vp))
        ranks = onp.arange(1, K + 1)
        pr = ranks ** (-cfg["zipf"]); self.prior = pr / pr.sum()
        base = rng.uniform(0.5, 1.5, K) * cfg["generic_rate"] * cfg["generic_mult"]
        self.rate = onp.array([min(base[r], 1.0) if VG50[r] in PARENT else 0.0 for r in range(K)])
        self.parent = onp.array([PID[PARENT[VG50[r]]] if VG50[r] in PARENT else -1 for r in range(K)])
        self.build_kg(rng)

    def build_kg(self, rng):
        cfg = self.cfg
        n_extra = 200
        self.n_ent = N_CLS + N_TYPES + n_extra
        ISA, NONVIS0 = K, K + 1  # relation ids: 0..49 predicates, 50 isA, 51..55 non-visual
        self.n_rel = K + 6
        trip = set()
        for c in range(N_CLS):
            trip.add((c, ISA, N_CLS + self.cls_type[c]))
        self.type_edge = onp.zeros((K, N_TYPES, N_TYPES), bool)
        self.concept_edge = onp.zeros((K, N_CLS, N_CLS), bool)
        n_correct = 0
        for r in range(K):
            for a, b in self.allowed[r]:
                if rng.random() < cfg["type_p"]:
                    self.type_edge[r, a, b] = True
                    trip.add((N_CLS + a, r, N_CLS + b))
            cov = min(rng.uniform(cfg["cov_lo"], cfg["cov_hi"]) * cfg["kg_cov_mult"], 1.0)
            vp = self.valid[r]
            take = vp[rng.random(len(vp)) < cov]
            for s, o in take:
                self.concept_edge[r, s, o] = True; trip.add((s, r, o))
            n_correct += len(take)
        n_wrong = int(round(cfg["wrong_frac"] * n_correct))
        k = 0
        while k < n_wrong:
            r = rng.integers(K); s, o = rng.integers(N_CLS, size=2)
            if s == o or self.concept_edge[r, s, o]:
                continue
            self.concept_edge[r, s, o] = True; trip.add((s, r, o)); k += 1
        k = 0
        while k < cfg["nonvis"]:
            s = rng.integers(self.n_ent); o = rng.integers(self.n_ent); r = NONVIS0 + rng.integers(5)
            if s != o and (s, r, o) not in trip:
                trip.add((s, r, o)); k += 1
        self.triples = onp.array(sorted(trip))
        # coverage count n_ij
        self.n_edges = self.concept_edge.sum(0).astype(float)
        te = self.type_edge.sum(0).astype(float)
        self.n_edges += 0.5 * te[self.cls_type][:, self.cls_type]


# --------------------------------------------------------------------------- data
def make_split(rng, W, n_img):
    """Returns dict of pair-level arrays for all ordered pairs of detected objects (SGDet)
    and the annotated GT triplets (PredCls)."""
    cfg = W.cfg
    pairs = []   # rows: img, gi, gj (gt obj ids, -1 for none), true_r, annot_r
    objs = []    # per object: img, true_cls, det(bool), pred_cls, conf
    obj_feat = []
    rel_feat = {}  # (img, gi, gj) -> (geo, uni)
    gt = []      # img, gs, go, true_r, annot_r
    oid = 0
    for im in range(n_img):
        n_rel = rng.integers(3, 8)
        im_objs = []
        def new_obj(c):
            nonlocal oid
            im_objs.append((oid, c)); oid += 1
            return oid - 1
        for _ in range(n_rel):
            r = rng.choice(K, p=W.prior)
            s, o = W.valid[r][rng.integers(len(W.valid[r]))]
            # subject reuse
            reuse = [x for x in im_objs if x[1] == s]
            gs = reuse[0][0] if reuse and rng.random() < 0.5 else new_obj(s)
            go = new_obj(o)
            ann = r
            if W.parent[r] >= 0 and rng.random() < W.rate[r]:
                ann = W.parent[r]
            vis = W.vis[r]
            g = vis * W.geo[r] + rng.normal(0, 1, D_GEO)
            u = vis * W.sig[r] + rng.normal(0, 1, D_UNI)
            gt.append((im, gs, go, r, ann, s, o))
            rel_feat[(gs, go)] = (g, u)
        for _ in range(rng.integers(1, 4)):
            new_obj(rng.integers(N_CLS))
        for (o_, c) in im_objs:
            det = rng.random() < cfg["det_recall"]
            if rng.random() < cfg["confusion"]:
                same = onp.where(W.cls_type == W.cls_type[c])[0]
                pc = rng.choice(same[same != c])
            else:
                pc = c
            conf = rng.beta(8, 2) if pc == c else rng.beta(4, 4)
            objs.append((im, c, det, pc, conf))
            obj_feat.append(W.W_reg @ W.lat[c] + rng.normal(0, 0.7, D_REG))
    objs = onp.array(objs, dtype=float)
    obj_feat = onp.array(obj_feat)
    gt = onp.array(gt)
    return dict(objs=objs, obj_feat=obj_feat, gt=gt, rel_feat=rel_feat, n_img=n_img)


def enumerate_pairs(sp, mode):
    objs = sp["objs"]; img = objs[:, 0].astype(int)
    keep = onp.ones(len(objs), bool) if mode == "gt" else objs[:, 2].astype(bool)
    idx = onp.where(keep)[0]
    I, J = [], []
    bounds = onp.r_[0, onp.nonzero(onp.diff(img[idx]))[0] + 1, len(idx)]
    for a, b in zip(bounds[:-1], bounds[1:]):
        lst = idx[a:b]
        ii, jj = onp.meshgrid(lst, lst, indexing="ij")
        m = ii != jj
        I.append(ii[m]); J.append(jj[m])
    return onp.concatenate(I), onp.concatenate(J)


def featurize(W, sp, I, J, mode, rng):
    objs = sp["objs"]
    lab = (objs[:, 1] if mode == "gt" else objs[:, 3]).astype(int)
    conf = onp.ones(len(objs)) if mode == "gt" else objs[:, 4]
    n = len(I)
    G = rng.normal(0, 1, (n, D_GEO)); U = rng.normal(0, 1, (n, D_UNI))
    rel = onp.zeros(n, bool)
    rf = sp["rel_feat"]
    for k in range(n):
        f = rf.get((I[k], J[k]))
        if f is not None:
            G[k], U[k] = f; rel[k] = True
    X = onp.concatenate([sp["obj_feat"][I], sp["obj_feat"][J], G, U,
                         W.lab_emb[lab[I]], W.lab_emb[lab[J]]], 1).astype(onp.float32)
    return dict(X=X, I=I, J=J, li=lab[I], lj=lab[J], qi=conf[I], qj=conf[J],
                img=objs[I, 0].astype(int), rel=rel)


# --------------------------------------------------------------------------- training helpers
def adam(params, gfun, batches, lr, epochs, rng, n):
    m = [onp.zeros_like(p) for p in params]; v = [onp.zeros_like(p) for p in params]
    t = 0
    for ep in range(epochs):
        perm = rng.permutation(n)
        for s in range(0, n, batches):
            b = perm[s:s + batches]
            gs = gfun(params, b)
            t += 1
            for k in range(len(params)):
                m[k] = 0.9 * m[k] + 0.1 * gs[k]
                v[k] = 0.999 * v[k] + 0.001 * gs[k] ** 2
                mh = m[k] / (1 - 0.9 ** t); vh = v[k] / (1 - 0.999 ** t)
                params[k] = params[k] - lr * mh / (onp.sqrt(vh) + 1e-8)
    return params


def init_mlp(rng, din, dh, dout):
    return [rng.normal(0, 1 / onp.sqrt(din), (din, dh)), onp.zeros(dh),
            rng.normal(0, 1 / onp.sqrt(dh), (dh, dout)), onp.zeros(dout)]


def encode(params, X):
    h = anp.tanh(anp.dot(X, params[0]) + params[1])
    z = anp.dot(h, params[2]) + params[3]
    return z / anp.sqrt(anp.sum(z ** 2, 1, keepdims=True) + 1e-9)


def train_proto(rng, X, y_local, text_seen, gamma=0.1, mu=1.0, epochs=14):
    """y_local indexes rows of text_seen (the trainable seen prototypes)."""
    params = init_mlp(rng, X.shape[1], 128, D_JOINT) + [text_seen.copy()]
    T0 = text_seen.copy()

    def loss(p, b):
        z = encode(p, X[b])
        P = p[4] / anp.sqrt(anp.sum(p[4] ** 2, 1, keepdims=True))
        logits = anp.dot(z, P.T) / gamma
        lse = anp.log(anp.sum(anp.exp(logits - anp.max(logits, 1, keepdims=True)), 1)) + anp.max(logits, 1)
        ce = anp.mean(lse - logits[onp.arange(len(b)), y_local[b]])
        return ce + mu * anp.sum((p[4] - T0) ** 2)
    g = grad(loss)
    params = adam(params, g, 256, 2e-3, epochs, rng, len(X))
    P = params[4] / onp.linalg.norm(params[4], axis=1, keepdims=True)
    return params[:4], P


def train_closed(rng, X, y, bias, epochs=14):
    params = init_mlp(rng, X.shape[1], 128, len(S_IDX))

    def loss(p, b):
        h = anp.tanh(anp.dot(X[b], p[0]) + p[1])
        lg = anp.dot(h, p[2]) + p[3] + bias[b]
        lse = anp.log(anp.sum(anp.exp(lg - anp.max(lg, 1, keepdims=True)), 1)) + anp.max(lg, 1)
        return anp.mean(lse - lg[onp.arange(len(b)), y[b]])
    return adam(params, grad(loss), 256, 2e-3, epochs, rng, len(X))


def closed_logits(p, X):
    return onp.tanh(X @ p[0] + p[1]) @ p[2] + p[3]


def train_related(rng, Xtr, ytr, Xva, yva):
    params = init_mlp(rng, Xtr.shape[1], 64, 1)

    def loss(p, b):
        h = anp.tanh(anp.dot(Xtr[b], p[0]) + p[1])
        s = (anp.dot(h, p[2]) + p[3])[:, 0]
        return anp.mean(anp.logaddexp(0, s) - ytr[b] * s)
    g = grad(loss)
    best, bestp, bad = 1e9, None, 0
    for ep in range(30):
        params = adam(params, g, 512, 2e-3, 1, rng, len(Xtr))
        s = closed_logits(params, Xva)[:, 0]
        vl = onp.mean(onp.logaddexp(0, s) - yva * s)
        if vl < best - 1e-4:
            best, bestp, bad = vl, [q.copy() for q in params], 0
        else:
            bad += 1
            if bad >= 3:
                break
    return bestp


def related_prob(p, X):
    return 1 / (1 + onp.exp(-closed_logits(p, X)[:, 0]))


def train_kge(rng, triples, n_ent, n_rel, kind="complex", dim=32, epochs=60):
    if kind == "complex":
        params = [rng.normal(0, 0.1, (n_ent, dim)), rng.normal(0, 0.1, (n_ent, dim)),
                  rng.normal(0, 0.1, (n_rel, dim)), rng.normal(0, 0.1, (n_rel, dim))]
    else:
        params = [rng.normal(0, 0.1, (n_ent, dim)), rng.normal(0, 0.1, (n_rel, dim))]
    h, r, t = triples[:, 0], triples[:, 1], triples[:, 2]
    score = kge_score(kind)
    st = {}

    def loss(p, b):
        nh = onp.where(st["side"][b], h[b], st["neg"][b]); nt = onp.where(st["side"][b], st["neg"][b], t[b])
        sp_ = score(p, h[b], r[b], t[b]); sn = score(p, nh, r[b], nt)
        reg = sum(anp.mean(q ** 2) for q in p)
        return anp.mean(anp.logaddexp(0, -sp_)) + anp.mean(anp.logaddexp(0, sn)) + 1e-2 * reg
    g = grad(loss)
    m = [onp.zeros_like(p) for p in params]; v = [onp.zeros_like(p) for p in params]; tt = 0
    n = len(triples)
    for ep in range(epochs):
        st["neg"] = rng.integers(n_ent, size=n); st["side"] = rng.random(n) < 0.5
        perm = rng.permutation(n)
        for s in range(0, n, 1024):
            b = perm[s:s + 1024]; gs = g(params, b); tt += 1
            for k in range(len(params)):
                m[k] = 0.9 * m[k] + 0.1 * gs[k]; v[k] = 0.999 * v[k] + 0.001 * gs[k] ** 2
                params[k] = params[k] - 1e-2 * (m[k] / (1 - 0.9 ** tt)) / (onp.sqrt(v[k] / (1 - 0.999 ** tt)) + 1e-8)
    return kind, params, score


def kge_score(kind):
    def score(p, hh, rr, tt):
        if kind == "complex":
            hr, hi, rr_, ri = p[0][hh], p[1][hh], p[2][rr], p[3][rr]
            tr, ti = p[0][tt], p[1][tt]
            return anp.sum(hr * rr_ * tr + hi * rr_ * ti + hr * ri * ti - hi * ri * tr, 1)
        return anp.sum(p[0][hh] * p[1][rr] * p[0][tt], 1)
    return score


# --------------------------------------------------------------------------- channels
def kg_scores(W, kge, li, lj, alpha=1.0, beta=1.5, eta=4.0, coverage_on=True):
    kind, p, score = kge
    n = len(li)
    S = onp.zeros((n, K))
    for r in range(K):
        phi = score(p, li, onp.full(n, r), lj)
        conc = W.concept_edge[r, li, lj].astype(float)
        typ = W.type_edge[r, W.cls_type[li], W.cls_type[lj]].astype(float)
        S[:, r] = eta * (alpha / (1 + onp.exp(-phi)) + beta * (conc + 0.5 * typ))
    c = 1 - onp.exp(-W.n_edges[li, lj] / 2.0)
    return S, c


def kapt(text, z_tr, li_tr, lj_tr, y_tr_global, Pfull, target_idx, seen_idx, CE, gamma=0.1, lam=0.5):
    """Move prototypes of target_idx toward weighted mean of KG-linked training pairs."""
    logits = z_tr @ Pfull[seen_idx].T / gamma
    pr = sp_softmax(logits, 1)
    pos = {g: k for k, g in enumerate(seen_idx)}
    py = pr[onp.arange(len(y_tr_global)), [pos[g] for g in y_tr_global]]
    w = 1 - py
    P = Pfull.copy()
    info = {}
    for r in target_idx:
        A = CE[r, li_tr, lj_tr]
        info[r] = int(A.sum())
        if A.sum() < 3:
            continue
        a = (w[A, None] * z_tr[A]).sum(0) / w[A].sum()
        m = (1 - lam) * text[r] + lam * a / onp.linalg.norm(a)
        P[r] = m / onp.linalg.norm(m)
    return P, info



def evidential(sA, sB, c, TA, TB, EA, EB):
    eA = EA * sp_softmax(sA / TA, 1)
    eB = EB * c[:, None] * sp_softmax(sB / TB, 1)
    SA = eA.sum(1, keepdims=True) + K; SB = eB.sum(1, keepdims=True) + K
    bA, bB = eA / SA, eB / SB; uA, uB = K / SA, K / SB
    C = bA.sum(1, keepdims=True) * bB.sum(1, keepdims=True) - (bA * bB).sum(1, keepdims=True)
    b = (bA * bB + bA * uB + bB * uA) / (1 - C)
    u = uA * uB / (1 - C)
    P = b + u / K
    return P / P.sum(1, keepdims=True)


def logit_adjust(sA, prior_seen_idx, prior, tau):
    lp = onp.log(prior[prior_seen_idx])
    shift = onp.zeros(K)
    shift[prior_seen_idx] = tau * (lp - lp.mean())
    return sA - shift


def zerorel(sA, sB, TA, TB, zeta, xi, tail):
    pA = sp_softmax(sA / TA, 1); pB = sp_softmax(sB / TB, 1)
    s = onp.maximum(pA, pB)
    top3A = onp.argsort(-pA, 1)[:, :3]; top3B = onp.argsort(-pB, 1)[:, :3]
    agree = onp.zeros_like(s)
    rows = onp.arange(len(s))[:, None]
    mA = onp.zeros_like(s, bool); mA[rows, top3A] = True
    mB = onp.zeros_like(s, bool); mB[rows, top3B] = True
    agree[mA & mB] = 1.0
    s = s * (1 + zeta * tail[None, :]) + xi * agree * pA
    return s / s.sum(1, keepdims=True)


def temp_softmax(s, T):
    return sp_softmax(s / T, 1)


# --------------------------------------------------------------------------- metrics
TGRID = onp.exp(onp.linspace(onp.log(0.05), onp.log(20), 40))
EGRID = [5, 10, 20, 50, 100, 200, 500, 1000]


def nll(P, y):
    return -onp.mean(onp.log(onp.clip(P[onp.arange(len(y)), y], 1e-12, 1)))


def predcls_metrics(P, y):
    pred = P.argmax(1)
    acc = (pred == y).mean()
    f1s = []
    for r in onp.unique(y):
        tp = ((pred == r) & (y == r)).sum(); fp = ((pred == r) & (y != r)).sum(); fn = ((pred != r) & (y == r)).sum()
        f1s.append(2 * tp / max(2 * tp + fp + fn, 1))
    conf = P.max(1); corr = (pred == y).astype(float)
    bins = onp.linspace(0, 1, 16); ece = 0
    for a, b in zip(bins[:-1], bins[1:]):
        m = (conf > a) & (conf <= b)
        if m.any():
            ece += m.mean() * abs(conf[m].mean() - corr[m].mean())
    Y = onp.zeros_like(P); Y[onp.arange(len(y)), y] = 1
    brier = ((P - Y) ** 2).sum(1).mean()
    bal = onp.mean([(pred[y == r] == r).mean() for r in onp.unique(y)])
    return dict(Acc=acc, MacroF1=onp.mean(f1s), ECE=ece, NLL=nll(P, y), Brier=brier, BalAcc=bal)


def sgdet_hits(Pd, pdat, rho, gt, n_img, ks=(20, 50, 100)):
    """Returns for each k a boolean hit vector over gt triplets (annotated labels)."""
    score = (pdat["qi"] * pdat["qj"] * rho)[:, None] * Pd
    top2 = onp.argsort(-score, 1)[:, :2]
    rows = onp.repeat(onp.arange(len(score)), 2)
    preds = top2.ravel(); sc = score[rows, preds]
    imgs = pdat["img"][rows]
    order = onp.lexsort((-sc, imgs))
    rows, preds, imgs = rows[order], preds[order], imgs[order]
    # rank within image
    first = onp.r_[0, onp.nonzero(onp.diff(imgs))[0] + 1]
    rank = onp.arange(len(imgs)) - onp.repeat(first, onp.diff(onp.r_[first, len(imgs)]))
    hits = {}
    # build set of (I,J,r) per k
    for k in ks:
        sel = rank < k
        keyset = set(zip(pdat["I"][rows[sel]], pdat["J"][rows[sel]], preds[sel],
                         pdat["li"][rows[sel]], pdat["lj"][rows[sel]]))
        hits[k] = onp.array([(int(g[1]), int(g[2]), int(g[4]), int(g[5]), int(g[6])) in keyset for g in gt])
    return hits


def sgdet_metrics(hits, gt_ann):
    out = {}
    unseen = onp.isin(gt_ann, U_IDX)
    for k, h in hits.items():
        out[f"R@{k}"] = h.mean()
        out[f"mR@{k}"] = onp.mean([h[gt_ann == r].mean() for r in onp.unique(gt_ann)])
        out[f"zR@{k}"] = h[unseen].mean()
    s = hits[100][~unseen].mean(); z = hits[100][unseen].mean()
    out["HM@100"] = 2 * s * z / (s + z + 1e-12)
    return out


In [ ]:
%%writefile run.py
import sys, time, json, pickle
import numpy as onp
from scipy.special import softmax as sm
sys.path.insert(0, ".")
from kare import *

TAU_GRID = [0, 0.25, 0.5, 0.75, 1.0]
ZETA, XI = [0, 0.25, 0.5, 1, 2], [0, 0.25, 0.5, 1]
EPS_UNSEEN = 1e-6


def coord_search(fn, grids, init, y, rounds=3):
    cur = dict(init)
    best = nll(fn(**cur), y)
    for _ in range(rounds):
        changed = False
        for k, g in grids.items():
            for v in g:
                trial = dict(cur); trial[k] = v
                l = nll(fn(**trial), y)
                if l < best - 1e-9:
                    best, cur, changed = l, trial, True
        if not changed:
            break
    return cur, best


def tune_T(s, y):
    ls = [nll(sm(s / T, 1), y) for T in TGRID]
    return TGRID[int(onp.argmin(ls))]


def embed_seen_only(p_seen, seen_idx):
    P = onp.full((len(p_seen), K), EPS_UNSEEN)
    P[:, seen_idx] = p_seen
    return P / P.sum(1, keepdims=True)


def run_seed(seed, cfg_over=None, full=True, log=print):
    t0 = time.time()
    cfg = dict(DEFAULT); cfg.update(cfg_over or {})
    rng = onp.random.default_rng(seed)
    W = World(rng, cfg)
    tr = make_split(rng, W, cfg["n_train"]); va = make_split(rng, W, cfg["n_val"]); te = make_split(rng, W, cfg["n_test"])
    stats = {}

    # ---------------- training triplets (unseen-annotated removed)
    def gt_feats(sp, keep_mask):
        g = sp["gt"][keep_mask]
        F = featurize(W, sp, g[:, 1].astype(int), g[:, 2].astype(int), "gt", rng)
        return g, F
    gtr = tr["gt"]
    keep_tr = ~onp.isin(gtr[:, 4], U_IDX)
    g_tr, F_tr = gt_feats(tr, keep_tr)
    stats["train_triplets"] = int(keep_tr.sum())
    unseen_true = onp.isin(gtr[:, 3], U_IDX)
    stats["unseen_generic_frac"] = float((gtr[unseen_true, 4] != gtr[unseen_true, 3]).mean())
    stats["kg_triples"] = int(len(W.triples))
    mu_, sd_ = F_tr["X"].mean(0), F_tr["X"].std(0) + 1e-6
    norm = lambda X: ((X - mu_) / sd_).astype(onp.float32)
    Xtr = norm(F_tr["X"]); ytr = g_tr[:, 4].astype(int)
    pos = {g: k for k, g in enumerate(S_IDX)}
    ytr_loc = onp.array([pos[v] for v in ytr])
    counts = onp.bincount(ytr, minlength=K).astype(float)
    prior_emp = (counts + 1e-3) / (counts + 1e-3).sum()

    # ---------------- relatedness head (shared)
    def rel_data(sp, n_neg):
        I, J = enumerate_pairs(sp, "gt")
        ann = {(int(a), int(b)) for a, b, l in zip(sp["gt"][:, 1], sp["gt"][:, 2], sp["gt"][:, 4]) if l not in U_IDX}
        y = onp.array([(int(a), int(b)) in ann for a, b in zip(I, J)], float)
        neg = onp.where(y == 0)[0]; posi = onp.where(y == 1)[0]
        neg = rng.choice(neg, min(n_neg, len(neg)), replace=False)
        idx = onp.r_[posi, neg]
        F = featurize(W, sp, I[idx], J[idx], "gt", rng)
        return norm(F["X"]), y[idx]
    Xr, yr = rel_data(tr, 120000); Xrv, yrv = rel_data(va, 20000)
    relp = train_related(rng, Xr, yr, Xrv, yrv)

    # ---------------- prototype channel (main) and auxiliary (pseudo-zero-shot)
    enc, Pseen = train_proto(rng, Xtr, ytr_loc, W.text[S_IDX])
    P_frozen = W.text.copy(); P_frozen[S_IDX] = Pseen
    z_tr = encode(enc, Xtr)
    P_kapt, kinfo = kapt(W.text, z_tr, g_tr[:, 5].astype(int), g_tr[:, 6].astype(int), ytr, P_frozen, U_IDX, S_IDX, W.concept_edge)
    stats["A_r_sizes"] = {VG50[r]: v for r, v in kinfo.items()}

    SA_aux = onp.array([i for i in S_IDX if i not in PS_IDX])
    m_aux = ~onp.isin(ytr, PS_IDX)
    pos_a = {g: k for k, g in enumerate(SA_aux)}
    enc_a, Pa = train_proto(rng, Xtr[m_aux], onp.array([pos_a[v] for v in ytr[m_aux]]), W.text[SA_aux])
    Pa_frozen = W.text.copy(); Pa_frozen[SA_aux] = Pa
    za = encode(enc_a, Xtr[m_aux])
    Pa_kapt, _ = kapt(W.text, za, g_tr[m_aux, 5].astype(int), g_tr[m_aux, 6].astype(int), ytr[m_aux], Pa_frozen, PS_IDX, SA_aux, W.concept_edge)
    counts_a = counts.copy(); counts_a[PS_IDX] = 0

    # ---------------- KG channel
    kges = {"complex": train_kge(rng, W.triples, W.n_ent, W.n_rel, "complex")}
    if full:
        kges["distmult"] = train_kge(rng, W.triples, W.n_ent, W.n_rel, "distmult")

    # ---------------- closed-set baselines
    tcl = W.cls_type
    def freq_table(cnt_mask):
        Cp = onp.zeros((N_CLS, N_CLS, K)); Ct = onp.zeros((N_TYPES, N_TYPES, K))
        li, lj, y = g_tr[cnt_mask, 5].astype(int), g_tr[cnt_mask, 6].astype(int), ytr[cnt_mask]
        onp.add.at(Cp, (li, lj, y), 1); onp.add.at(Ct, (tcl[li], tcl[lj], y), 1)
        return Cp, Ct
    Cp, Ct = freq_table(onp.ones(len(ytr), bool))
    gprior = counts[S_IDX] / counts[S_IDX].sum()
    def freq_prob(li, lj):
        ct = Ct[tcl[li], tcl[lj]][:, S_IDX]
        pt = (ct + gprior) / (ct.sum(1, keepdims=True) + 1)
        cp = Cp[li, lj][:, S_IDX]
        return (cp + 5 * pt) / (cp.sum(1, keepdims=True) + 5)
    bias_tr = onp.log(freq_prob(g_tr[:, 5].astype(int), g_tr[:, 6].astype(int)))
    closed = train_closed(rng, Xtr, ytr_loc, bias_tr)
    geo_cols = onp.arange(2 * D_REG, 2 * D_REG + D_GEO + D_UNI)

    # ---------------- scoring helper for any pair set
    def channel_scores(X, li, lj):
        z = encode(enc, X); za_ = encode(enc_a, X)
        out = dict(sA_frozen=z @ P_frozen.T / 0.1, sA_kapt=z @ P_kapt.T / 0.1,
                   aux_frozen=za_ @ Pa_frozen.T / 0.1, aux_kapt=za_ @ Pa_kapt.T / 0.1)
        for k, kg in kges.items():
            out[f"sB_{k}"], out["c"] = kg_scores(W, kg, li, lj)
        fp = freq_prob(li, lj)
        out["freq"] = fp
        lg = closed_logits(closed, X)
        out["motifs"] = lg + onp.log(fp)
        Xcf = X.copy(); Xcf[:, geo_cols] = 0.0
        out["tde"] = lg - closed_logits(closed, Xcf)
        return out

    # ---------------- validation (all seen-annotated triplets)
    keep_va = ~onp.isin(va["gt"][:, 4], U_IDX)
    g_va, F_va = gt_feats(va, keep_va)
    Xva = norm(F_va["X"]); yva = g_va[:, 4].astype(int)
    Sv = channel_scores(Xva, g_va[:, 5].astype(int), g_va[:, 6].astype(int))

    tunings = {}
    # simple temperature baselines on main models (seen val)
    yloc_va = onp.array([pos[v] for v in yva])
    tunings["Proto"] = tune_T(Sv["sA_frozen"], yva)
    tunings["KG-only"] = tune_T(Sv["sB_complex"], yva)
    tunings["Motifs"] = tune_T(Sv["motifs"][:, :], yloc_va)
    tunings["TDE"] = tune_T(Sv["tde"], yloc_va)

    # pseudo-zero-shot fusion tuning
    tail_main = tail_from(counts); tail_aux = tail_from(counts_a)
    fusion_cfgs = {}

    def tune_kare(sA_key, sB_key, coverage=True, taus=TAU_GRID):
        res = []
        c = Sv["c"] if coverage else onp.ones(len(yva))
        for tau in taus:
            sA = logit_adjust(Sv[sA_key], SA_aux, prior_from(counts_a), tau)
            fn = lambda TA, TB, EA, EB: evidential(sA, Sv[sB_key], c, TA, TB, EA, EB)
            best, l = coord_search(fn, dict(TA=TGRID, TB=TGRID, EA=EGRID, EB=EGRID),
                                   dict(TA=1.0, TB=1.0, EA=50, EB=50), yva)
            P = fn(**best)
            bal = onp.mean([(P[yva == r].argmax(1) == r).mean() for r in onp.unique(yva)])
            res.append(dict(tau=tau, nll=l, bal=bal, **best))
        return res

    def tune_zr(sA_key):
        fn = lambda TA, TB, zeta, xi: zerorel(Sv[sA_key], Sv["sB_complex"], TA, TB, zeta, xi, tail_aux)
        best, l = coord_search(fn, dict(TA=TGRID, TB=TGRID, zeta=ZETA, xi=XI),
                               dict(TA=1.0, TB=1.0, zeta=0, xi=0), yva)
        return dict(nll=l, **best)

    rk = tune_kare("aux_kapt", "sB_complex")
    fusion_cfgs["KARE"] = dict(kind="ev", sA="sA_kapt", sB="sB_complex", cov=True, **min(rk, key=lambda d: d["nll"]))
    fusion_cfgs["KARE-B"] = dict(kind="ev", sA="sA_kapt", sB="sB_complex", cov=True, **max(rk, key=lambda d: d["bal"]))
    fusion_cfgs["ZeroRel"] = dict(kind="zr", sA="sA_frozen", **tune_zr("aux_frozen"))
    tunings["KARE_tau_curve"] = rk
    r = tune_kare("aux_kapt", "sB_complex", coverage=False)
    fusion_cfgs["KARE w/o coverage"] = dict(kind="ev", sA="sA_kapt", sB="sB_complex", cov=False, **min(r, key=lambda d: d["nll"]))
    if full:
        r = tune_kare("aux_frozen", "sB_complex")
        fusion_cfgs["KARE w/o KAPT"] = dict(kind="ev", sA="sA_frozen", sB="sB_complex", cov=True, **min(r, key=lambda d: d["nll"]))
        r = tune_kare("aux_kapt", "sB_complex", taus=[0])
        fusion_cfgs["KARE w/o logit adj."] = dict(kind="ev", sA="sA_kapt", sB="sB_complex", cov=True, **r[0])
        r = tune_kare("aux_kapt", "sB_distmult")
        fusion_cfgs["KARE (DistMult KG)"] = dict(kind="ev", sA="sA_kapt", sB="sB_distmult", cov=True, **min(r, key=lambda d: d["nll"]))
        fusion_cfgs["ZeroRel + KAPT"] = dict(kind="zr", sA="sA_kapt", **tune_zr("aux_kapt"))
    tunings["fusion"] = fusion_cfgs

    def predict_all(S):
        out = {}
        out["Freq"] = embed_seen_only(S["freq"], S_IDX)
        out["Motifs-style"] = embed_seen_only(sm(S["motifs"] / tunings["Motifs"], 1), S_IDX)
        out["TDE-style"] = embed_seen_only(sm(S["tde"] / tunings["TDE"], 1), S_IDX)
        out["Proto"] = sm(S["sA_frozen"] / tunings["Proto"], 1)
        out["KG-only"] = sm(S["sB_complex"] / tunings["KG-only"], 1)
        for name, f in fusion_cfgs.items():
            if f["kind"] == "zr":
                out[name] = zerorel(S[f["sA"]], S["sB_complex"], f["TA"], f["TB"], f["zeta"], f["xi"], tail_main)
            else:
                sA = logit_adjust(S[f["sA"]], S_IDX, prior_from(counts), f["tau"])
                c = S["c"] if f["cov"] else onp.ones(len(S["c"]))
                out[name] = evidential(sA, S[f["sB"]], c, f["TA"], f["TB"], f["EA"], f["EB"])
        return out

    # ---------------- test PredCls (all annotated test triplets)
    g_te, F_te = gt_feats(te, onp.ones(len(te["gt"]), bool))
    Ste = channel_scores(norm(F_te["X"]), g_te[:, 5].astype(int), g_te[:, 6].astype(int))
    yte = g_te[:, 4].astype(int)
    Pte = predict_all(Ste)
    results = {m: predcls_metrics(P, yte) for m, P in Pte.items()}
    per_img_pc = {m: dict(correct=(P.argmax(1) == yte), nll=-onp.log(onp.clip(P[onp.arange(len(yte)), yte], 1e-12, 1)))
                  for m, P in Pte.items()}

    # ---------------- test SGDet
    I, J = enumerate_pairs(te, "det")
    stats["test_pairs"] = int(len(I))
    Fd = featurize(W, te, I, J, "det", rng)
    Xd = norm(Fd["X"])
    rho = related_prob(relp, Xd)
    Sd = channel_scores(Xd, Fd["li"], Fd["lj"])
    Pd = predict_all(Sd)
    hits = {}
    for m, P in Pd.items():
        h = sgdet_hits(P, Fd, rho, te["gt"], te["n_img"])
        hits[m] = h
        results[m].update(sgdet_metrics(h, te["gt"][:, 4].astype(int)))
    # coverage diagnostics on test GT pairs
    cov_gt = 1 - onp.exp(-W.n_edges[g_te[:, 5].astype(int), g_te[:, 6].astype(int)] / 2.0)
    stats["zero_cov_frac_test_gt"] = float((cov_gt == 0).mean())
    stats["seconds"] = time.time() - t0
    log(f"seed {seed} done in {stats['seconds']:.0f}s")
    return dict(results=results, stats=stats, tunings=tunings,
                bootstrap=dict(hits={m: {k: v for k, v in h.items()} for m, h in hits.items()},
                               gt_img=te["gt"][:, 0].astype(int), gt_ann=te["gt"][:, 4].astype(int),
                               pc=per_img_pc, pc_img=g_te[:, 0].astype(int), pc_y=yte, pc_true=g_te[:, 3].astype(int),
                               cov_gt=cov_gt, P_te={m: Pte[m] for m in ["Proto", "KG-only", "ZeroRel", "KARE"]}))


def tail_from(counts):
    order = onp.argsort(-counts, kind="stable")
    rank = onp.empty(K); rank[order] = onp.arange(K)
    t = rank / (K - 1)
    t[counts == 0] = 1.0
    return t


def prior_from(counts):
    return (counts + 1e-3) / (counts + 1e-3).sum()


if __name__ == "__main__":
    seed = int(sys.argv[1]); full = sys.argv[2] == "full" if len(sys.argv) > 2 else True
    over = json.loads(sys.argv[3]) if len(sys.argv) > 3 else {}
    tag = sys.argv[4] if len(sys.argv) > 4 else f"main_{seed}"
    out = run_seed(seed, over, full)
    pickle.dump(out, open(f"out/{tag}.pkl", "wb"))
    import pandas as pd
    print(pd.DataFrame(out["results"]).T.round(4).to_string())
    print(json.dumps(out["stats"], default=str)[:1500])


In [ ]:
%%writefile aggregate.py
import pickle, sys, json
import numpy as onp, pandas as pd
sys.path.insert(0, ".")
from kare import U_IDX, VG50, K

seeds = [0, 1, 2, 3, 4]
runs = [pickle.load(open(f"out/main_{s}.pkl", "rb")) for s in seeds]
METRICS = ["R@20", "R@50", "R@100", "mR@20", "mR@50", "mR@100", "zR@20", "zR@50", "zR@100", "HM@100",
           "Acc", "MacroF1", "ECE", "NLL", "Brier"]
methods = list(runs[0]["results"].keys())
mean = pd.DataFrame({m: {k: onp.mean([r["results"][m][k] for r in runs]) for k in METRICS} for m in methods}).T
sd = pd.DataFrame({m: {k: onp.std([r["results"][m][k] for r in runs], ddof=1) for k in METRICS} for m in methods}).T
pd.set_option("display.width", 250)
print("MEAN\n", mean.round(4).to_string()); print("SD\n", sd.round(4).to_string())

# ---- paired bootstrap over pooled test images
rng = onp.random.default_rng(12345)
B = 2000
img_off = 0
H = {m: [] for m in methods}; Nn = []; PC = {m: [] for m in methods}; PCn = []; PCnll = {m: [] for m in methods}
for r in runs:
    bs = r["bootstrap"]; n_img = bs["gt_img"].max() + 1
    gi, ga = bs["gt_img"], bs["gt_ann"]
    N = onp.zeros((1500, K)); onp.add.at(N, (gi, ga), 1); Nn.append(N)
    for m in methods:
        Hm = onp.zeros((1500, K)); onp.add.at(Hm, (gi, ga), bs["hits"][m][100].astype(float)); H[m].append(Hm)
        c = onp.zeros(1500); onp.add.at(c, bs["pc_img"], bs["pc"][m]["correct"].astype(float)); PC[m].append(c)
        l = onp.zeros(1500); onp.add.at(l, bs["pc_img"], bs["pc"][m]["nll"]); PCnll[m].append(l)
    cn = onp.zeros(1500); onp.add.at(cn, bs["pc_img"], 1); PCn.append(cn)
N = onp.vstack(Nn); PCn = onp.concatenate(PCn)
H = {m: onp.vstack(v) for m, v in H.items()}; PC = {m: onp.concatenate(v) for m, v in PC.items()}
PCnll = {m: onp.concatenate(v) for m, v in PCnll.items()}
n_img = N.shape[0]
W = rng.multinomial(n_img, onp.ones(n_img) / n_img, size=B).astype(float)  # B x n_img
seen = onp.ones(K, bool); seen[U_IDX] = False


def stats(m, w):
    h = w @ H[m]; n = w @ N
    R = h.sum(1) / n.sum(1)
    with onp.errstate(invalid="ignore", divide="ignore"):
        rr = h / n
    mR = onp.nanmean(onp.where(n > 0, rr, onp.nan), 1)
    zR = h[:, ~seen].sum(1) / n[:, ~seen].sum(1)
    sR = h[:, seen].sum(1) / n[:, seen].sum(1)
    HM = 2 * sR * zR / (sR + zR)
    acc = (w @ PC[m]) / (w @ PCn); nl = (w @ PCnll[m]) / (w @ PCn)
    return dict(R100=R, mR100=mR, zR100=zR, HM100=HM, Acc=acc, NLL=nl)


ones = onp.ones((1, n_img))
point = {m: {k: float(v[0]) for k, v in stats(m, ones).items()} for m in methods}
bstat = {m: stats(m, W) for m in ["KARE", "KARE-B", "ZeroRel", "Proto", "KG-only", "Motifs-style"]}
rows = []
for a, b in [("KARE", "ZeroRel"), ("KARE", "Proto"), ("KARE", "KG-only"), ("KARE-B", "ZeroRel"), ("KARE", "Motifs-style")]:
    for k in ["R100", "mR100", "zR100", "HM100", "Acc", "NLL"]:
        d = bstat[a][k] - bstat[b][k]; d0 = point[a][k] - point[b][k]
        lo, hi = onp.percentile(d, [2.5, 97.5])
        p = 2 * min((d <= 0).mean(), (d >= 0).mean())
        rows.append(dict(cmp=f"{a} vs {b}", metric=k, diff=d0, lo=lo, hi=hi, p=max(p, 1 / B)))
boot = pd.DataFrame(rows)
print(boot.round(4).to_string())

# ---- per-unseen-predicate zR@100 pooled
pp = {}
for m in ["Proto", "KG-only", "ZeroRel", "KARE", "KARE-B"]:
    h = H[m].sum(0); n = N.sum(0)
    pp[m] = {VG50[r]: h[r] / n[r] for r in U_IDX}
pp = pd.DataFrame(pp); pp["n_test"] = [int(N.sum(0)[r]) for r in U_IDX]
pp["A_r_mean"] = [onp.mean([r["stats"]["A_r_sizes"][VG50[u]] for r in runs]) for u in U_IDX]
print(pp.round(3).to_string())

# ---- PredCls accuracy on unseen GT by coverage bin
cov_rows = []
for r in runs:
    bs = r["bootstrap"]; y = bs["pc_y"]; cov = bs["cov_gt"]
    un = onp.isin(y, U_IDX)
    for m in ["KG-only", "ZeroRel", "KARE"]:
        P = bs["P_te"][m]
        for name, msk in [("c = 0", cov == 0), ("0 < c < 0.4", (cov > 0) & (cov < 0.4)), ("c >= 0.4", cov >= 0.4)]:
            mm = un & msk
            cov_rows.append(dict(method=m, bin=name, n=int(mm.sum()), acc=(P[mm].argmax(1) == y[mm]).mean() if mm.any() else onp.nan))
cov = pd.DataFrame(cov_rows).groupby(["bin", "method"]).agg(n=("n", "sum"), acc=("acc", "mean"))
print(cov.round(3).to_string())

st = pd.DataFrame([r["stats"] for r in runs]).drop(columns=["A_r_sizes"])
print(st.to_string())
tau = [(r["tunings"]["fusion"]["KARE"]["tau"], r["tunings"]["fusion"]["KARE-B"]["tau"]) for r in runs]
print("tau KARE/KARE-B per seed:", tau)
print("KARE fusion params:", [{k: v for k, v in r["tunings"]["fusion"]["KARE"].items() if k in ("TA", "TB", "EA", "EB")} for r in runs])
pickle.dump(dict(mean=mean, sd=sd, boot=boot, pp=pp, cov=cov, st=st, tau=tau), open("out/agg.pkl", "wb"))


In [ ]:
%%writefile sweep.py
import subprocess, sys, json, os
specs = []
for f in [0.1, 0.25, 0.5, 1.0]:
    for s in [0, 1, 2]:
        specs.append((f"kg_{f}_{s}", s, {"kg_cov_mult": f, "type_p": round(0.7 * f, 4)}))
for g in [0.0, 0.5, 1.0, 1.5, 2.0]:
    for s in [0, 1, 2]:
        specs.append((f"gen_{g}_{s}", s, {"generic_mult": g}))
lo, hi = int(sys.argv[1]), int(sys.argv[2])
for tag, s, cfg in specs[lo:hi]:
    if os.path.exists(f"out/{tag}.pkl"): continue
    r = subprocess.run(["python3", "run.py", str(s), "lite", json.dumps(cfg), tag], capture_output=True, text=True, timeout=900)
    print(tag, "ok" if r.returncode == 0 else r.stderr[-500:], flush=True)


In [ ]:
%%writefile agg_sweep.py
import pickle, numpy as onp, pandas as pd
rows=[]
for kind, vals in [("kg",[0.1,0.25,0.5,1.0]),("gen",[0.0,0.5,1.0,1.5,2.0])]:
    for v in vals:
        for s in [0,1,2]:
            r=pickle.load(open(f"out/{kind}_{v}_{s}.pkl","rb"))
            for m in ["Proto","KG-only","ZeroRel","KARE","KARE w/o coverage"]:
                R=r["results"][m]
                rows.append(dict(kind=kind,v=v,seed=s,method=m,zR=R["zR@100"],HM=R["HM@100"],R100=R["R@100"],Acc=R["Acc"],NLL=R["NLL"],
                                 zc=r["stats"]["zero_cov_frac_test_gt"],gfrac=r["stats"]["unseen_generic_frac"],kgt=r["stats"]["kg_triples"]))
d=pd.DataFrame(rows)
pd.set_option("display.width",250)
g=d.groupby(["kind","v","method"])[["zR","HM","R100","Acc","NLL"]].agg(["mean","std"]).round(3)
print(g.to_string())
print(d.groupby(["kind","v"])[["zc","gfrac","kgt"]].mean().round(3))
# paired diff coverage vs no coverage
for kind in ["kg","gen"]:
    x=d[d.kind==kind].pivot_table(index=["v","seed"],columns="method",values=["zR","HM","Acc","NLL"])
    for met in ["zR","HM","Acc","NLL"]:
        diff=(x[(met,"KARE")]-x[(met,"KARE w/o coverage")]).groupby("v").agg(["mean","min","max"])
        print(kind,met,"KARE - w/o cov\n",diff.round(4).to_string())
d.to_pickle("out/sweep.pkl")


In [ ]:
%%writefile extra_analyses.py
"""Analyses added in the extended paper: Tables 1, 3 (selected values), 10, 11, 13 and the Spearman test in Section 5.3."""
import pickle, sys, numpy as np, pandas as pd
sys.path.insert(0, ".")
from kare import U_IDX
from scipy.stats import spearmanr
from scipy.special import softmax
runs = [pickle.load(open(f"out/main_{s}.pkl", "rb")) for s in range(5)]

def ece(P, y):
    conf, corr, e = P.max(1), (P.argmax(1) == y), 0.0
    edges = np.linspace(0, 1, 16)
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b)
        if m.any(): e += m.mean() * abs(conf[m].mean() - corr[m].mean())
    return e

# Table 10: seen vs unseen
rows = []
for m in ["Proto", "KG-only", "ZeroRel", "KARE"]:
    r = {"method": m}
    for part in ["seen", "unseen"]:
        acc, ec, nl = [], [], []
        for run in runs:
            b = run["bootstrap"]; y = b["pc_y"]; P = b["P_te"][m]; un = np.isin(y, U_IDX)
            k = un if part == "unseen" else ~un
            acc.append((P[k].argmax(1) == y[k]).mean()); ec.append(ece(P[k], y[k]))
            nl.append(-np.log(np.clip(P[k][np.arange(k.sum()), y[k]], 1e-12, 1)).mean())
        r[f"{part}_acc"], r[f"{part}_ece"], r[f"{part}_nll"] = np.mean(acc), np.mean(ec), np.mean(nl)
    r["seen_R100"] = np.mean([run["bootstrap"]["hits"][m][100][~np.isin(run["bootstrap"]["gt_ann"], U_IDX)].mean() for run in runs])
    r["unseen_mass_on_seen"] = np.mean([run["bootstrap"]["P_te"][m][~np.isin(run["bootstrap"]["pc_y"], U_IDX)][:, U_IDX].sum(1).mean() for run in runs])
    rows.append(r)
print("Table 10\n", pd.DataFrame(rows).round(3).to_string())

# Table 11: tau curve on pseudo-zero-shot validation
tc = pd.DataFrame([(d["tau"], d["nll"], d["bal"]) for run in runs for d in run["tunings"]["KARE_tau_curve"]], columns=["tau", "nll", "bal"])
print("Table 11\n", tc.groupby("tau").agg(["mean", "std"]).round(3))

# Table 3: selected fusion values
for run in runs:
    f = run["tunings"]["fusion"]
    print({k: {kk: (round(vv, 3) if isinstance(vv, float) else vv) for kk, vv in f[k].items() if kk in ("TA", "TB", "EA", "EB", "tau", "zeta", "xi")} for k in ["KARE", "KARE-B", "ZeroRel"]})

# Section 5.3: anchor size vs gain
pp = pickle.load(open("out/agg.pkl", "rb"))["pp"]
print("Spearman |A_r| vs KARE-ZeroRel gain:", spearmanr(pp["A_r_mean"], pp["KARE"] - pp["ZeroRel"]))

# Table 13: per seed
for s, run in enumerate(runs):
    st, R = run["stats"], run["results"]
    print(s, st["train_triplets"], st["kg_triples"], st["test_pairs"], f"{100*st['unseen_generic_frac']:.1f}%", f"{100*st['zero_cov_frac_test_gt']:.1f}%",
          *[round(R[m][k], 3) for m in ["ZeroRel", "KARE"] for k in ["R@100", "zR@100", "HM@100", "Acc"]])

# Table 1: worked fusion example (K = 4)
def fuse(eA, eB, K=4):
    SA, SB = eA.sum() + K, eB.sum() + K; bA, bB, uA, uB = eA / SA, eB / SB, K / SA, K / SB
    C = bA.sum() * bB.sum() - (bA * bB).sum(); b = (bA * bB + bA * uB + bB * uA) / (1 - C); u = uA * uB / (1 - C)
    return bB, uB, C, b + u / K
eA = np.array([12., 6., 1., 1.]); kg = np.array([2., 16., 1., 1.])
for c in [0, 0.2, 0.8]:
    bB, uB, C, P = fuse(eA, c * kg); print(f"c={c}: bB(riding)={bB[1]:.2f} uB={uB:.2f} C={C:.2f} P={P.round(3)}")
m = np.maximum(eA / eA.sum(), np.full(4, 0.25)); print("max rule with uniform KG:", (m / m.sum()).round(3))

# Section 5.7: label-noise analysis (needs pc_true, recorded by run.py)
for m in ["Proto", "KG-only", "ZeroRel", "KARE"]:
    a1, a2, a3 = [], [], []
    for run in runs:
        b = run["bootstrap"]; P = b["P_te"][m]; y = b["pc_y"]; t = b["pc_true"]; pr = P.argmax(1)
        hid = (y != t) & np.isin(t, U_IDX)
        a1.append((pr == y).mean()); a2.append((pr == t).mean()); a3.append((pr[hid] == t[hid]).mean())
    print(f"{m}: accuracy vs annotation {np.mean(a1):.3f}, vs true predicate {np.mean(a2):.3f}, hidden unseen recovered {np.mean(a3):.3f}")
c = n = 0
for run in runs:
    b = run["bootstrap"]; y, t = b["pc_y"], b["pc_true"]; s = ~np.isin(y, U_IDX)
    pk, pz = b["P_te"]["KARE"].argmax(1), b["P_te"]["ZeroRel"].argmax(1)
    k = s & (pk != y) & (pz == y); n += k.sum(); c += (pk[k] == t[k]).sum()
    if run is runs[0]: pass
print(f"KARE-wrong/ZeroRel-right seen triplets where KARE matches the true predicate: {c}/{n} = {c/n:.3f}")
leak = {m: np.concatenate([r["bootstrap"]["P_te"][m][~np.isin(r["bootstrap"]["pc_y"], U_IDX)][:, U_IDX].sum(1) for r in runs]) for m in ["ZeroRel", "KARE"]}
for m, v in leak.items(): print(f"{m}: mass on unseen for seen pairs median {np.median(v):.3f}, <0.1 {np.mean(v<0.1):.3f}, >0.5 {np.mean(v>0.5):.3f}")


## 2. Main experiment: five seeds, all baselines and ablations

In [ ]:
for s in range(5):
    !python run.py {s} full '{{}}' main_{s}

## 3. Tables 4–9 and the paired bootstrap

In [ ]:
!python aggregate.py

## 4. Robustness sweeps (Figures 2–3) and the genericity-zero ablation (Section 5.2)

In [ ]:
!python sweep.py 0 27
!python agg_sweep.py
for s in [0, 1]:
    !python run.py {s} full '{"generic_mult":0.0}' genfull_{s}

### Figures 2–3

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
d = pd.read_pickle('out/sweep.pkl')
for kind, xlab, labels in [('kg', 'Fraction of KG predicate edges kept', {0.1:'10%',0.25:'25%',0.5:'50%',1.0:'100%'}),
                           ('gen', 'Unseen interactions labelled with their coarse parent', {0.0:'0%',0.5:'21%',1.0:'42%',1.5:'62%',2.0:'81%'})]:
    fig, ax = plt.subplots(figsize=(7, 4))
    for m in ['KARE', 'KG-only', 'ZeroRel', 'Proto']:
        g = d[(d.kind == kind) & (d.method == m)].groupby('v')['zR'].agg(['mean', 'std'])
        ax.errorbar(range(len(g)), g['mean'], yerr=g['std'], marker='o', capsize=3, label=m, lw=2.5 if m == 'KARE' else 1.2)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels.values()); ax.set_xlabel(xlab); ax.set_ylabel('zR@100'); ax.set_ylim(0, 0.5); ax.legend(); plt.show()

## 5. Additional analyses: Tables 1, 3, 10, 11, 13 and the Spearman test

In [ ]:
!python extra_analyses.py

## 6. Publication figures (Figures 2–11)
Writes 300-dpi PNGs to `figs/`. Figure 1 is the method diagram and is not data-driven.

In [ ]:
%%writefile figures.py
"""Figures 2-11 of the paper, drawn from the saved experiment outputs in out/."""
import pickle, sys, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, ".")
from kare import U_IDX, VG50

OUT = "figs"
import os; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
                     "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight"})
COL = {"Freq": "#bdbdbd", "Motifs-style": "#8c8c8c", "TDE-style": "#5a5a5a", "Proto": "#7b5ea7", "KG-only": "#3a923a",
       "ZeroRel": "#e08214", "KARE": "#1f5fbf", "KARE-B": "#7fa8e6"}
runs = [pickle.load(open(f"out/main_{s}.pkl", "rb")) for s in range(5)]
agg = pickle.load(open("out/agg.pkl", "rb")); mean, sd = agg["mean"], agg["sd"]
MAIN = ["Freq", "Motifs-style", "TDE-style", "Proto", "KG-only", "ZeroRel", "KARE", "KARE-B"]

# ---- Figure 2: main comparison
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={"width_ratios": [1.35, 1]})
mets = ["R@100", "mR@100", "zR@100", "HM@100"]; w = 0.1
for k, m in enumerate(MAIN):
    x = np.arange(len(mets)) + (k - 3.5) * w
    axes[0].bar(x, mean.loc[m, mets], w, yerr=sd.loc[m, mets], color=COL[m], label=m, capsize=1.5, error_kw={"lw": 0.6})
axes[0].set_xticks(range(len(mets))); axes[0].set_xticklabels(mets); axes[0].set_ylabel("Recall (SGDet)")
axes[0].set_title("(a) Scene graph detection"); axes[0].set_ylim(0, 0.6)
mets2 = ["Acc", "MacroF1", "ECE"]
for k, m in enumerate(MAIN):
    x = np.arange(len(mets2)) + (k - 3.5) * w
    axes[1].bar(x, mean.loc[m, mets2], w, yerr=sd.loc[m, mets2], color=COL[m], capsize=1.5, error_kw={"lw": 0.6})
axes[1].set_xticks(range(len(mets2))); axes[1].set_xticklabels(["Accuracy", "Macro-F1", "ECE ↓"])
axes[1].set_title("(b) Predicate classification"); axes[1].set_ylim(0, 0.85)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=8, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.savefig(f"{OUT}/fig2_main.png"); plt.close(fig)

# ---- Figure 3: seen/unseen trade-off and zR@k
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
ax = axes[0]
for m in ["Motifs-style", "Proto", "KG-only", "ZeroRel", "KARE", "KARE-B"]:
    pts = []
    for r in runs:
        b = r["bootstrap"]; h = b["hits"][m][100]; un = np.isin(b["gt_ann"], U_IDX)
        pts.append((h[~un].mean(), h[un].mean()))
    pts = np.array(pts)
    ax.scatter(pts[:, 0], pts[:, 1], s=14, color=COL[m], alpha=0.55, lw=0)
    ax.scatter(*pts.mean(0), s=60, color=COL[m], edgecolor="k", lw=0.6, label=m, zorder=3)
s = np.linspace(0.05, 0.6, 200)
for hm in [0.1, 0.2, 0.3, 0.4]:
    z = hm * s / (2 * s - hm); ok = (z > 0) & (z < 0.5)
    ax.plot(s[ok], z[ok], ls=":", lw=0.7, color="grey"); ax.text(s[ok][-1], z[ok][-1], f" HM={hm}", fontsize=6.5, color="grey", va="center")
ax.set_xlabel("Seen-predicate R@100"); ax.set_ylabel("Zero-shot recall zR@100"); ax.set_xlim(0.2, 0.62); ax.set_ylim(-0.02, 0.5)
ax.set_title("(a) Seen versus unseen recall (5 seeds)"); ax.legend(loc="upper right", fontsize=6.8, frameon=False)
ax = axes[1]
for m in ["Proto", "KG-only", "ZeroRel", "KARE", "KARE-B"]:
    y = [mean.loc[m, f"zR@{k}"] for k in (20, 50, 100)]; e = [sd.loc[m, f"zR@{k}"] for k in (20, 50, 100)]
    ax.errorbar([20, 50, 100], y, yerr=e, marker="o", ms=3.5, color=COL[m], label=m, capsize=2, lw=2.2 if m == "KARE" else 1.2)
ax.set_xticks([20, 50, 100]); ax.set_xlabel("k (triplets kept per image)"); ax.set_ylabel("zR@k"); ax.set_ylim(0, 0.5)
ax.set_title("(b) Zero-shot recall as the budget grows"); ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_tradeoff.png"); plt.close(fig)

# ---- Figure 4: ablations
AB = ["ZeroRel", "ZeroRel + KAPT", "KARE w/o KAPT", "KARE w/o coverage", "KARE w/o logit adj.", "KARE (DistMult KG)", "KARE"]
lab = ["ZeroRel", "ZeroRel + KAPT", "KARE w/o KAPT", "KARE w/o coverage", "KARE w/o logit adj.", "KARE, DistMult KG", "KARE (full)"]
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), sharey=True)
for ax, met, ttl in zip(axes, ["zR@100", "HM@100", "Acc"], ["(a) zR@100", "(b) HM@100", "(c) PredCls accuracy"]):
    c = ["#1f5fbf" if a == "KARE" else ("#e08214" if a.startswith("ZeroRel") else "#9db7de") for a in AB]
    ax.barh(range(len(AB)), mean.loc[AB, met], xerr=sd.loc[AB, met], color=c, capsize=2, error_kw={"lw": 0.6})
    ax.axvline(mean.loc["KARE", met], color="#1f5fbf", ls="--", lw=0.7); ax.set_title(ttl)
axes[0].set_yticks(range(len(AB))); axes[0].set_yticklabels(lab)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_ablation.png"); plt.close(fig)

# ---- Figure 5: per-predicate zR@100
pp = agg["pp"].sort_values("KARE")
fig, ax = plt.subplots(figsize=(7.2, 2.8)); w = 0.2
for k, m in enumerate(["Proto", "KG-only", "ZeroRel", "KARE"]):
    ax.bar(np.arange(len(pp)) + (k - 1.5) * w, pp[m], w, color=COL[m], label=m)
ax.set_xticks(range(len(pp))); ax.set_xticklabels([f"{p}\nn={n}\n|A|={a:.0f}" for p, n, a in zip(pp.index, pp["n_test"], pp["A_r_mean"])], fontsize=7)
ax.set_ylabel("zR@100 (pooled, 5 seeds)"); ax.legend(ncol=4, frameon=False, loc="upper left"); ax.set_ylim(0, 0.55)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_perpred.png"); plt.close(fig)

# ---- Figure 6: accuracy by coverage bin
cov = agg["cov"].reset_index()
bins = ["c = 0", "0 < c < 0.4", "c >= 0.4"]
fig, ax = plt.subplots(figsize=(4.6, 2.5)); w = 0.25
for k, m in enumerate(["KG-only", "ZeroRel", "KARE"]):
    v = [cov[(cov.bin == b) & (cov.method == m)]["acc"].values[0] for b in bins]
    ax.bar(np.arange(3) + (k - 1) * w, v, w, color=COL[m], label=m)
n = [int(cov[(cov.bin == b) & (cov.method == "KARE")]["n"].values[0]) for b in bins]
ax.set_xticks(range(3)); ax.set_xticklabels([f"{b.replace('>=', '≥')}\n(n={x})" for b, x in zip(bins, n)])
ax.set_ylabel("Accuracy on unseen triplets"); ax.legend(frameon=False); ax.set_ylim(0, 0.7)
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_coverage.png"); plt.close(fig)

# ---- Figures 7-8: robustness sweeps (Word version of the doc's interactive charts)
d = pd.read_pickle("out/sweep.pkl")
for kind, fn, xl, labels in [("kg", "fig7_kg", "Fraction of KG predicate edges kept", ["10%", "25%", "50%", "100%"]),
                             ("gen", "fig8_gen", "Unseen-predicate training interactions labelled with their coarse parent", ["0%", "21%", "42%", "62%", "81%"])]:
    fig, ax = plt.subplots(figsize=(5.2, 2.8))
    for m in ["Proto", "KG-only", "ZeroRel", "KARE"]:
        g = d[(d.kind == kind) & (d.method == m)].groupby("v")["zR"].agg(["mean", "std"])
        ax.errorbar(range(len(g)), g["mean"], yerr=g["std"], marker="o", ms=3.5, capsize=2, color=COL[m], label=m, lw=2.2 if m == "KARE" else 1.2)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels); ax.set_xlabel(xl); ax.set_ylabel("zR@100 (3 seeds)"); ax.set_ylim(0, 0.5)
    ax.legend(frameon=False, ncol=2, fontsize=7.5)
    fig.tight_layout(); fig.savefig(f"{OUT}/{fn}.png"); plt.close(fig)

# ---- Figure 9: reliability diagrams (PredCls, all test triplets pooled)
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.1), sharey=True)
edges = np.linspace(0, 1, 11)
for ax, m in zip(axes, ["Proto", "KG-only", "ZeroRel", "KARE"]):
    P = np.vstack([r["bootstrap"]["P_te"][m] for r in runs]); y = np.concatenate([r["bootstrap"]["pc_y"] for r in runs])
    conf, corr = P.max(1), (P.argmax(1) == y)
    accs, confs, cnt = [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        k = (conf > a) & (conf <= b)
        accs.append(corr[k].mean() if k.any() else np.nan); confs.append(conf[k].mean() if k.any() else np.nan); cnt.append(k.sum())
    ax.bar(edges[:-1] + 0.05, accs, 0.09, color=COL[m], alpha=0.85)
    ax.plot([0, 1], [0, 1], "k--", lw=0.6); ax.set_title(f"{m}  (ECE {mean.loc[m, 'ECE']:.3f})", fontsize=8.5)
    ax.set_xlabel("Confidence"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[0].set_ylabel("Accuracy")
fig.tight_layout(); fig.savefig(f"{OUT}/fig9_reliability.png"); plt.close(fig)

# ---- Figure 10: where the accuracy cost comes from
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
for m in ["ZeroRel", "KARE"]:
    v = np.concatenate([r["bootstrap"]["P_te"][m][~np.isin(r["bootstrap"]["pc_y"], U_IDX)][:, U_IDX].sum(1) for r in runs])
    ax.hist(v, bins=np.linspace(0, 1, 26), color=COL[m], alpha=0.6, label=f"{m} (mean {v.mean():.2f})", density=True)
ax.set_xlabel("Probability mass on the ten unseen predicates"); ax.set_ylabel("Density"); ax.set_title("(a) Leakage on seen-predicate pairs")
ax.legend(frameon=False)
ax = axes[1]; w = 0.2
ms = ["Proto", "KG-only", "ZeroRel", "KARE"]
for k, m in enumerate(ms):
    sa, ua = [], []
    for r in runs:
        b = r["bootstrap"]; y = b["pc_y"]; P = b["P_te"][m]; un = np.isin(y, U_IDX)
        sa.append((P[~un].argmax(1) == y[~un]).mean()); ua.append((P[un].argmax(1) == y[un]).mean())
    ax.bar(np.arange(2) + (k - 1.5) * w, [np.mean(sa), np.mean(ua)], w, yerr=[np.std(sa, ddof=1), np.std(ua, ddof=1)], color=COL[m], label=m, capsize=2, error_kw={"lw": 0.6})
ax.set_xticks([0, 1]); ax.set_xticklabels(["Seen triplets (95.4%)", "Unseen triplets (4.6%)"]); ax.set_ylabel("PredCls accuracy")
ax.set_title("(b) Accuracy by predicate group"); ax.legend(frameon=False, fontsize=7); ax.set_ylim(0, 0.85)
fig.tight_layout(); fig.savefig(f"{OUT}/fig10_cost.png"); plt.close(fig)

# ---- Figure 11: tau trade-off on the pseudo-zero-shot validation split
tc = pd.DataFrame([(d_["tau"], d_["nll"], d_["bal"]) for r in runs for d_ in r["tunings"]["KARE_tau_curve"]], columns=["tau", "nll", "bal"]).groupby("tau").agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(4.6, 2.6)); ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
ax.errorbar(tc.index, tc["nll"]["mean"], yerr=tc["nll"]["std"], marker="o", ms=3.5, color="#1f5fbf", capsize=2, label="Validation NLL")
ax2.errorbar(tc.index, tc["bal"]["mean"], yerr=tc["bal"]["std"], marker="s", ms=3.5, color="#e08214", capsize=2, label="Balanced accuracy")
ax.axvline(0, color="#1f5fbf", ls=":", lw=0.8); ax.axvline(0.5, color="#e08214", ls=":", lw=0.8)
ax.text(0.02, 2.15, "KARE", color="#1f5fbf", fontsize=7.5); ax.text(0.52, 2.15, "KARE-B", color="#e08214", fontsize=7.5)
ax.set_xlabel("Logit-adjustment strength τ"); ax.set_ylabel("Validation NLL", color="#1f5fbf"); ax2.set_ylabel("Balanced accuracy", color="#e08214")
fig.tight_layout(); fig.savefig(f"{OUT}/fig11_tau.png"); plt.close(fig)
print("done", sorted(os.listdir(OUT)))


In [ ]:
!python figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("figs/*.png")): display(Image(f, width=700))